In [5]:
from pathlib import Path
import pandas as pd
#leer los resultados 

BASE = Path("/home/lka/HYBRID_SARIMA")
# Rutas de SARIMA
SARIMA_PATH = BASE / 'results' 
sarima_results = pd.read_csv(SARIMA_PATH / 'SARIMA' / 'sarima_IC_test.csv')
lstm_results = pd.read_csv(SARIMA_PATH / 'LSTM' / 'lstm_IC_test.csv')
gru_results = pd.read_csv(SARIMA_PATH / 'GRU' / 'gru_IC_test.csv')
sarima_lstm_seq_results = pd.read_csv(SARIMA_PATH / 'SARIMA_LSTM_Sequential' / 'sarima_lstm_IC_test.csv')
sarima_gru_seq_results = pd.read_csv(SARIMA_PATH / 'SARIMA_GRU_Sequential' / 'sarima_gru_IC_test.csv')

In [6]:
# Academic style
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
sns.set_theme(style="whitegrid", context="paper", font_scale=1.3)

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.labelsize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "pdf.fonttype": 42,
    "savefig.bbox": "tight"
})

In [ ]:
import ast
from matplotlib.patches import Patch


# Model results
data = {
    "SARIMA": sarima_results,
    "LSTM": lstm_results,
    "GRU": gru_results,
    "SARIMA-LSTM_seq": sarima_lstm_seq_results,
    "SARIMA-GRU_seq": sarima_gru_seq_results
}

models = list(data.keys())

# Abbreviated model names
model_names = {
    "SARIMA": "SARIMA",
    "LSTM": "LSTM",
    "GRU": "GRU",
    "SARIMA-LSTM_seq": r"SL$_{\mathrm{seq}}$",
    "SARIMA-GRU_seq": r"SG$_{\mathrm{seq}}$"
}

series = [
    "15-34 M", "15-34 F",
    "35-49 M", "35-49 F",
    "50-74 M", "50-74 F",
    "Global Rate"
]

metrics = ["MAE", "MSE", "RMSE", "MAPE"]

# Combine all datasets
df = pd.concat([
    frame.assign(Model=model)
    for model, frame in data.items()
], ignore_index=True)

df = df.rename(columns={"Serie": "Series"})
df["Series"] = df["Series"].str.replace("_", " ", regex=False)

# Extract 95% confidence interval limits
for metric in metrics:
    bounds = df[f"{metric} IC 95%"].apply(ast.literal_eval)

    df[f"{metric}_lower"] = bounds.apply(lambda x: float(x[0]))
    df[f"{metric}_upper"] = bounds.apply(lambda x: float(x[1]))

# Output directory
OUT = BASE / "IMG" / "model_comparison"
OUT.mkdir(parents=True, exist_ok=True)

# Colors and positions
colors = sns.color_palette("Set2", n_colors=7)
x = np.arange(len(models))
width = 0.11

legend_items = [
    Patch(facecolor=colors[i], label=serie)
    for i, serie in enumerate(series)
]

# Generate one figure for each metric
for metric in metrics:

    fig, ax = plt.subplots(figsize=(14, 7))

    for i, serie in enumerate(series):

        subset = (
            df[df["Series"] == serie]
            .set_index("Model")
            .reindex(models)
        )

        values = subset[metric].to_numpy(dtype=float)
        lower = subset[f"{metric}_lower"].to_numpy(dtype=float)
        upper = subset[f"{metric}_upper"].to_numpy(dtype=float)

        # Asymmetric 95% confidence intervals
        errors = np.vstack([
            values - lower,
            upper - values
        ])

        positions = x + (i - 3) * width

        ax.bar(
            positions,
            values,
            width=width,
            color=colors[i],
            yerr=errors,
            capsize=3,
            error_kw={
                "elinewidth": 1.1,
                "capthick": 1.1,
                "ecolor": "black"
            }
        )

    # Axes
    ax.set_xticks(x)
    ax.set_xticklabels([model_names[m] for m in models])
    ax.set_xlabel("")

    ax.set_ylabel(
        "MAPE (%)" if metric == "MAPE" else metric
    )

    ax.set_ylim(bottom=0)
    ax.grid(axis="y", linestyle=":", alpha=0.5)
    ax.grid(axis="x", visible=False)
    ax.set_axisbelow(True)

    sns.despine(ax=ax)

    # Legend above, centered, 2 rows
    fig.legend(
        handles=legend_items,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.98),
        ncol=4,
        frameon=False,
        columnspacing=2.5,
        handletextpad=0.6,
        fontsize=18
    )

    # Space between legend and figure
    fig.subplots_adjust(
        top=0.76,
        bottom=0.13,
        left=0.09,
        right=0.98
    )

    # Export to PDF for Overleaf
    plt.savefig(
        OUT / f"{metric.lower()}_test_ic95.pdf",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

print("All metric figures generated successfully.")


'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp
'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp
'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp
'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp


All metric figures generated successfully.


In [ ]:
# Model results
data = {
    "SARIMA": sarima_results,
    "LSTM": lstm_results,
    "GRU": gru_results,
    "SARIMA-LSTM_seq": sarima_lstm_seq_results,
    "SARIMA-GRU_seq": sarima_gru_seq_results
}

# Abbreviated model names
model_names = {
    "SARIMA": "SARIMA",
    "LSTM": "LSTM",
    "GRU": "GRU",
    "SARIMA-LSTM_seq": r"SL$_{\mathrm{seq}}$",
    "SARIMA-GRU_seq": r"SG$_{\mathrm{seq}}$"
}

models = list(model_names.values())

series = [
    "15-34 M", "15-34 F",
    "35-49 M", "35-49 F",
    "50-74 M", "50-74 F",
    "Global Rate"
]

metrics = ["MAE", "MSE", "RMSE", "MAPE"]

# Combine all datasets
df = pd.concat([
    frame.assign(Model=model)
    for model, frame in data.items()
], ignore_index=True)

df = df.rename(columns={"Serie": "Series"})
df["Series"] = df["Series"].str.replace("_", " ", regex=False)

# Replace model names
df["Model"] = df["Model"].replace(model_names)

# Extract 95% confidence interval limits
for metric in metrics:
    bounds = df[f"{metric} IC 95%"].apply(ast.literal_eval)

    df[f"{metric}_lower"] = bounds.apply(lambda x: float(x[0]))
    df[f"{metric}_upper"] = bounds.apply(lambda x: float(x[1]))

# Output directory
OUT = BASE / "IMG" / "model_comparison"
OUT.mkdir(parents=True, exist_ok=True)

# Colors and positions
colors = sns.color_palette("Set2", n_colors=7)
x = np.arange(len(models))
width = 0.11

legend_items = [
    Patch(facecolor=colors[i], label=serie)
    for i, serie in enumerate(series)
]

# Generate ONE figure with four subplots
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

for ax, metric in zip(axes.flat, metrics):

    for i, serie in enumerate(series):

        subset = (
            df[df["Series"] == serie]
            .set_index("Model")
            .reindex(models)
        )

        values = subset[metric].to_numpy(dtype=float)
        lower = subset[f"{metric}_lower"].to_numpy(dtype=float)
        upper = subset[f"{metric}_upper"].to_numpy(dtype=float)

        # Asymmetric 95% confidence intervals
        errors = np.vstack([
            values - lower,
            upper - values
        ])

        positions = x + (i - 3) * width

        ax.bar(
            positions,
            values,
            width=width,
            color=colors[i],
            yerr=errors,
            capsize=3,
            error_kw={
                "elinewidth": 1.1,
                "capthick": 1.1,
                "ecolor": "black"
            }
        )

    # Axes formatting
    ax.set_xticks(x)
    ax.set_xticklabels(models)
    ax.set_xlabel("")
    ax.set_ylabel("MAPE (%)" if metric == "MAPE" else metric)
    ax.set_ylim(bottom=0)

    ax.grid(axis="y", linestyle=":", alpha=0.5)
    ax.grid(axis="x", visible=False)
    ax.set_axisbelow(True)

    sns.despine(ax=ax)

# One shared legend for all four plots
fig.legend(
    handles=legend_items,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.98),
    ncol=4,
    frameon=False,
    columnspacing=2.5,
    handletextpad=0.6,
    fontsize=18
)

# Space between legend and plots
fig.subplots_adjust(
    top=0.82,
    bottom=0.10,
    left=0.09,
    right=0.98,
    hspace=0.15,
    wspace=0.10
)

# Export combined figure
plt.savefig(
    OUT / "all_metrics_test_ic95.pdf",
    dpi=300,
    bbox_inches="tight"
)
plt.close()

print("All metric figures generated successfully.")

'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp


All metric figures generated successfully.
